
# 02 . Bronze - Schema Drift / Shape-Collapse Audit

Finds fields where inference silently collapsed structure - without already knowing where to look. Catches struct-nested fields; fields inside arrays need an explode first, out of scope here.

In [0]:
from pyspark.sql import functions as F

catalog = "novalake"
raw_path = f"/Volumes/{catalog}/bronze/landing/payments_events.json"
df_inferred = spark.read.json(raw_path)

In [0]:
def flatten_struct_leaves(schema, prefix=""):
    leaves = []
    for field in schema.fields:
        path = f"{prefix}.{field.name}" if prefix else field.name
        dtype = field.dataType
        if hasattr(dtype, "fields"):
            leaves.extend(flatten_struct_leaves(dtype, path))
        else:
            leaves.append((path, str(dtype)))
    return leaves

leaves = flatten_struct_leaves(df_inferred.schema)
string_leaves = [path for path, dtype in leaves if dtype == "StringType()"]
print(f"{len(string_leaves)} string-typed leaf field to check:")
for p in string_leaves:
    print("-", p)

In [0]:
def shape_profile(df, col_path):
    col = F.col(col_path)
    return(
        df.select(
            F.when(col.isNull(), "Null")
             .when(col.rlike(r'^\{.*\}$'), "json_object")
             .when(col.rlike(r'^\{.*\}$'), "json_array")
             .when(col.rlike(r'^-?\d+\.?\d*$'), "numeric")
             .when(col.rlike(r'^\d{4}-\d{2}-\d{2}'), "date_like")
             .otherwise("plain_text")
             .alias("shape")
        )
        .groupBy("shape").count()
        .orderBy(F.desc("count"))
    )


print("Fields with MORE THAN ONE non-null shape - investigate these:\n")
for path in string_leaves:
    profile = shape_profile(df_inferred, path)
    rows = profile.collect()
    nonnull_shapes = [r["shape"] for r in rows if r["shape"].lower() != "null"]
    if len(nonnull_shapes) > 1:
        print(f" {path}")
        profile.show()